# Lumpy 02: Forecasting

Notebook version: 3

This is the stripped-down replacement for the old full lumpy workbook.

Kept in the default path:

- Zero Forecast as the control
- SBA/Croston as the current winning intermittent-demand baseline
- Recent Mean 6m as the cheap demand-level challenger
- TSB as a cheap intermittent-demand comparator
- Phase-1 summaries for monthly totals, 3-month rolling WMAPE, and SKU-horizon allocation

Paused unless explicitly switched on:

- Hurdle Random Forest, because the current version is slow and underperforms the baselines
- Legacy Aggregate Allocation, because the current version over-allocates and should be replaced by the Phase 2 allocation work
- 6-month and 8-month lag sweeps, because the required 3-month lag is currently best on the SKU-horizon check

The archived full notebook is still available as `archive_lumpy_collision_spare_parts_forecasting_full.ipynb`.


In [ ]:
import time
KERNEL_START_TIME = time.perf_counter()
print("OK: kernel is running")


In [ ]:
from pathlib import Path
import importlib
import sys
import time

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [
        start,
        start.parent,
        start.parent.parent,
        start / "Inchscape Repo",
        start.parent / "Inchscape Repo",
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find repo root. Open this notebook from the Inchscape Repo folder "
        "or update find_repo_root() with the project path."
    )


root = find_repo_root()
src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

_lumpy = importlib.import_module("lumpy_forecasting")
_lumpy = importlib.reload(_lumpy)

DEFAULT_MODEL_NAMES = _lumpy.DEFAULT_MODEL_NAMES
LumpyConfig = _lumpy.LumpyConfig
best_model_by_window = _lumpy.best_model_by_window
build_lumpy_feature_inventory = _lumpy.build_lumpy_feature_inventory
build_lumpy_model_frame = _lumpy.build_lumpy_model_frame
build_lumpy_phase1_tables = _lumpy.build_lumpy_phase1_tables
load_lumpy_inputs = _lumpy.load_lumpy_inputs
make_backtest_splits_for_lags = _lumpy.make_backtest_splits_for_lags
run_lumpy_backtest = _lumpy.run_lumpy_backtest
summarize_by_model = _lumpy.summarize_by_model
summarize_monthly_totals = _lumpy.summarize_monthly_totals
write_lumpy_outputs = _lumpy.write_lumpy_outputs

print("OK: lumpy_forecasting imported")
print(f"Project root: {root}")
print(f"Module path: {Path(_lumpy.__file__).resolve()}")


In [ ]:
# Start in fast-check mode so VS Code proves the notebook is working quickly.
# Required benchmark: 18-month forecast window with a 3-month operational lag.
# Full mode compares 9/12/18-month windows, but keeps the required 3-month lag by default.
FAST_RUN = True
SAVE_OUTPUTS = not FAST_RUN

# Phase 2/diagnostic switches. Leave these off for the first pass; they add time
# and the current results show they are not beating SBA Croston.
RUN_DIAGNOSTIC_LAGS = False
RUN_LEGACY_AGGREGATE = False
RUN_RANDOM_FOREST = False

FORECAST_WINDOWS = (18,) if FAST_RUN else (9, 12, 18)
LAG_OPTIONS = (3, 6, 8) if RUN_DIAGNOSTIC_LAGS else (3,)
REQUIRED_WINDOW_MONTHS = 18
REQUIRED_LAG_MONTHS = 3

config = LumpyConfig(
    train_months=48,
    gap_months=REQUIRED_LAG_MONTHS,
    test_months=REQUIRED_WINDOW_MONTHS,
    step_months=3,
    min_train_months=18,
    max_folds=1 if FAST_RUN else None,
    external_mode="selected",
)

model_names = DEFAULT_MODEL_NAMES
if RUN_LEGACY_AGGREGATE:
    model_names = model_names + ("aggregate_allocation",)
if RUN_RANDOM_FOREST:
    model_names = model_names + ("hurdle_random_forest",)

print("Mode:", "FAST CHECK" if FAST_RUN else "FULL BACKTEST")
print("Models:", model_names)
print("Forecast windows:", FORECAST_WINDOWS)
print("Lag options:", LAG_OPTIONS)
print("Required benchmark:", f"{REQUIRED_WINDOW_MONTHS}m window / {REQUIRED_LAG_MONTHS}m lag")
print("Diagnostic lag sweep:", RUN_DIAGNOSTIC_LAGS)
print("Legacy aggregate allocation:", RUN_LEGACY_AGGREGATE)
print("Hurdle random forest:", RUN_RANDOM_FOREST)
print("Max folds per window/lag:", config.max_folds)
print("Save outputs:", SAVE_OUTPUTS)


In [ ]:
start_time = time.perf_counter()
print("Loading lumpy inputs and building model frame...")

sales, external = load_lumpy_inputs(root, config)
model_data, external_inventory = build_lumpy_model_frame(sales, external, config)
splits = make_backtest_splits_for_lags(
    model_data,
    config,
    gap_month_options=LAG_OPTIONS,
    test_month_options=FORECAST_WINDOWS,
)
feature_inventory = build_lumpy_feature_inventory(model_data, splits, config)

split_summary = (
    splits.groupby(["test_months", "gap_months"], as_index=False)
    .agg(
        folds=("fold_id", "count"),
        first_test_start=("test_start", "min"),
        last_test_end=("test_end", "max"),
        min_train_months=("train_months", "min"),
    )
    .sort_values(["test_months", "gap_months"])
)
required_benchmark_folds = splits.loc[
    splits["test_months"].eq(REQUIRED_WINDOW_MONTHS)
    & splits["gap_months"].eq(REQUIRED_LAG_MONTHS)
]

elapsed = time.perf_counter() - start_time
print(f"OK: data ready in {elapsed:.1f}s")
print("Sales rows:", len(sales))
print("Model rows:", len(model_data))
print("SKUs:", model_data["sku_id"].nunique())
print("Folds:", len(splits))
print("Required benchmark folds:", len(required_benchmark_folds))
print("External source/features:", len(external_inventory))
print("Model features:", len(feature_inventory))

print("Window/lag split summary:")
display(split_summary)

print("External handoff usage:")
display(external_inventory["usage"].value_counts().rename_axis("usage").reset_index(name="count"))

print("Model feature families, previewed on the required 18m/3m benchmark split:")
display(feature_inventory["feature_family"].value_counts().rename_axis("feature_family").reset_index(name="count"))

display(external_inventory)
display(feature_inventory)
display(splits)


In [ ]:
start_time = time.perf_counter()
print(f"Running {len(model_names)} model(s) over {len(splits)} fold(s)...")
print("This cell is the main work. In fast-check mode it should finish quickly.")

forecasts = run_lumpy_backtest(
    data=model_data,
    splits=splits,
    config=config,
    model_names=model_names,
)

phase1_tables = build_lumpy_phase1_tables(
    model_data=model_data,
    splits=splits,
    forecasts=forecasts,
)

model_summary = phase1_tables["model_summary"]
monthly_totals = phase1_tables["monthly_totals"]
best_models = phase1_tables["best_model"]
monthly_total_model_summary = phase1_tables["monthly_total_model_summary"]
sku_horizon_model_summary = phase1_tables["sku_horizon_model_summary"]
metric_suite_model_summary = phase1_tables["metric_suite_model_summary"]
lag_comparison_agent_report = phase1_tables["lag_comparison_agent_report"]

elapsed = time.perf_counter() - start_time
print(f"OK: backtest and phase-1 summaries finished in {elapsed:.1f}s")
print("Forecast rows:", len(phase1_tables["forecasts"]))
print("Phase-1 tables:", len(phase1_tables))

print("SKU-month WMAPE summary:")
display(model_summary)

print("Best SKU-month model by window:")
display(best_models)

print("Monthly-total summary - often the cleaner demand-planning lens:")
display(monthly_total_model_summary.sort_values(["window_label", "monthly_total_wmape_percent", "model"]))

print("SKU-horizon summary - SKU quantity across the forecast window, separate from exact month timing:")
display(sku_horizon_model_summary.sort_values(["window_label", "horizon_sku_wmape_percent", "model"]))

print("Phase-1 combined metric view:")
display(metric_suite_model_summary)

print("Lag comparison agent:")
display(lag_comparison_agent_report)


In [ ]:
if SAVE_OUTPUTS:
    output_paths = write_lumpy_outputs(
        root=root,
        model_data=model_data,
        splits=splits,
        forecasts=forecasts,
        external_inventory=external_inventory,
        feature_inventory=feature_inventory,
        phase1_tables=phase1_tables,
    )

    print("OK: saved lumpy outputs")
    display({name: str(path) for name, path in output_paths.items()})
else:
    print("FAST_RUN is on, so outputs were not overwritten.")
    print("Set FAST_RUN = False in the config cell, then rerun all cells to save final outputs.")
